# Q-Learning

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/reinforcement-learning/02-q-learning

A from-scratch, runnable implementation of the concepts in the lesson.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Tabular Q-learning, from scratch

Model-free: the agent never sees the transition rules — it learns $Q(s,a)$ purely from sampled experience using the temporal-difference update.

In [ ]:
class GridWorld:
    """n x n grid. Start top-left (0), goal bottom-right. Actions: 0=up 1=down 2=left 3=right.
    Reward -1 per step, +10 at the goal (terminal)."""
    def __init__(self, n=5):
        self.n = n; self.nS = n*n; self.nA = 4; self.goal = n*n-1
    def step(self, s, a):
        r, c = divmod(s, self.n)
        if a==0: r = max(0, r-1)
        elif a==1: r = min(self.n-1, r+1)
        elif a==2: c = max(0, c-1)
        else: c = min(self.n-1, c+1)
        s2 = r*self.n + c
        done = (s2 == self.goal)
        return s2, (10.0 if done else -1.0), done

env = GridWorld(5)

## The TD update + ε-greedy exploration

$Q(s,a)\leftarrow Q(s,a)+\alpha\big[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\big]$, behaving with a decaying ε-greedy policy.

In [ ]:
def q_learning(env, episodes=2000, alpha=0.5, gamma=0.9):
    rng = np.random.RandomState(0)
    Q = np.zeros((env.nS, env.nA))
    eps, returns = 1.0, []
    for ep in range(episodes):
        s, done, total, steps = 0, False, 0, 0
        while not done and steps < 100:
            a = rng.randint(env.nA) if rng.rand() < eps else int(np.argmax(Q[s]))
            s2, r, done = env.step(s, a)
            Q[s,a] += alpha * (r + gamma*np.max(Q[s2]) - Q[s,a])   # off-policy TD
            s = s2; total += r; steps += 1
        eps = max(0.05, eps*0.999); returns.append(total)
    return Q, returns

Q, returns = q_learning(env)
print('learned. final greedy policy reaches goal from start.')

## Learning curve and learned policy

In [ ]:
import numpy as np
ma = np.convolve(returns, np.ones(50)/50, mode='valid')
fig, ax = plt.subplots(1, 2, figsize=(13,5))
ax[0].plot(ma, color='#14b8a6'); ax[0].set_xlabel('episode'); ax[0].set_ylabel('return (50-ep avg)')
ax[0].set_title('Q-learning improves with experience')
arrows = {0:'↑',1:'↓',2:'←',3:'→'}
pi = np.argmax(Q, axis=1)
ax[1].imshow(Q.max(1).reshape(5,5), cmap='viridis')
for s in range(env.nS):
    r,c = divmod(s, env.n)
    ax[1].text(c, r, 'G' if s==env.goal else arrows[pi[s]], ha='center', va='center', color='w', fontsize=16)
ax[1].set_title('max Q (color) + greedy policy'); ax[1].axis('off'); plt.show()

## Off-policy in action

The update used $\max_{a'}Q(s',a')$ — the *best* next action — even though the agent often explored randomly. That is why Q-learning converges to the **optimal** policy while behaving sub-optimally.

In [ ]:
# Greedy rollout from the start with the learned Q
s, path, done = 0, [0], False
while not done and len(path) < 20:
    s, _, done = env.step(s, int(np.argmax(Q[s]))); path.append(s)
print('greedy path (state ids):', path)
print('reached goal:', path[-1] == env.goal, 'in', len(path)-1, 'steps (optimal = 8)')

## Key takeaways

- Q-learning learns $Q^*$ from sampled `(s,a,r,s')` with no model of the environment.
- The **TD update** bootstraps from the agent's own next estimate.
- It is **off-policy**: the target uses the best next action regardless of behavior.
- **ε-greedy** with decay balances exploration early and exploitation later.

## ✏️ Your turn

### Exercise 1 — Q-learning TD update

The Q-learning update rule:

$$Q(s,a) \\leftarrow Q(s,a) + \\alpha \\Big[r + \\gamma \\max_{a'} Q(s',a') - Q(s,a)\\Big]$$

The bracketed term is the **TD error** — the difference between the bootstrapped target and the current estimate. Implement it and verify on hand-checkable fixtures.

In [ ]:
def td_update(q_sa, alpha, r, gamma, max_q_next):
    """One Q-learning update. Returns the new Q(s,a) value."""
    # TODO(you): implement the formula above
    ...

In [ ]:
result = td_update(0.0, 0.5, -1.0, 0.9, 2.0)
assert abs(result - 0.4) < 1e-9, \
    "Q=0, r=-1, maxQ'=2, alpha=0.5 => 0 + 0.5*(-1+1.8-0) = 0.4"
assert abs(td_update(5.0, 0.5, 10.0, 0.9, 0.0) - 7.5) < 1e-9, \
    "goal step: Q=5, r=10, maxQ'=0 => 5 + 0.5*(10+0-5) = 7.5"
assert abs(td_update(2.0, 0.1, -1.0, 0.9, 2.0) - 1.88) < 1e-9, \
    "small alpha changes slowly: 2 + 0.1*(-1+1.8-2) = 1.88"
# with alpha=1, td_update fully replaces old value with target
assert abs(td_update(99.0, 1.0, 5.0, 0.9, 0.0) - 5.0) < 1e-9, \
    "alpha=1 fully replaces old Q with the target r+gamma*max_Q'"
# Zero reward everywhere: at Q=0, maxQ'=0, r=0 the TD error is 0 -- no change
assert abs(td_update(0.0, 0.5, 0.0, 0.9, 0.0) - 0.0) < 1e-9, \
    "zero reward everywhere, all Q=0: the TD error is 0, so nothing changes"
# Single-state MDP with a self-loop: the fixed point of the update satisfies
# Q* = r + gamma*Q*, i.e. Q* = r / (1 - gamma). Updating at that fixed point is a no-op.
r_fp, gamma_fp, alpha_fp = -1.0, 0.9, 0.5
q_star = r_fp / (1 - gamma_fp)
assert abs(td_update(q_star, alpha_fp, r_fp, gamma_fp, q_star) - q_star) < 1e-9, \
    "single-state self-loop MDP: the update is a no-op exactly at the fixed point"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def td_update(q_sa, alpha, r, gamma, max_q_next):
    td_error = r + gamma * max_q_next - q_sa
    return q_sa + alpha * td_error
```

</details>

### Exercise 2 — Epsilon-greedy exploration

With probability $\\varepsilon$ the agent picks a random action (exploration);
otherwise it picks $\\arg\\max_a Q(s, a)$ (exploitation).

Implement it and verify: at $\\varepsilon=0$ always greedy; at $\\varepsilon=1$ always random;
for intermediate $\\varepsilon$ the greedy action is chosen most often.

In [ ]:
import numpy as np

def epsilon_greedy(q_values, epsilon, seed=None):
    """Return an action index using epsilon-greedy selection.
    q_values: 1-D array of Q values for each action.
    epsilon: exploration probability in [0, 1].
    seed: optional random seed for reproducibility."""
    # TODO(you): with prob epsilon return a random action, else return argmax
    ...

In [ ]:
q = np.array([1.0, 5.0, 2.0])  # greedy action = 1

# epsilon=0: always greedy
assert all(epsilon_greedy(q, 0.0, seed=i) == 1 for i in range(20)), \
    "epsilon=0 must always pick the greedy (argmax) action"

# epsilon=1: uniformly random (verify all actions appear in 300 draws)
actions = [epsilon_greedy(q, 1.0, seed=i) for i in range(300)]
assert set(actions) == {0, 1, 2}, \
    "epsilon=1 must explore all actions"

# intermediate epsilon: greedy action chosen more than random (50 draws)
greedy_count = sum(epsilon_greedy(q, 0.2, seed=i) == 1 for i in range(50))
assert greedy_count > 30, \
    "with epsilon=0.2 the greedy action should be chosen most of the time"
# Zero-variance Q: all arms tied -- argmax always breaks ties by the first index
q_ties = np.array([2.0, 2.0, 2.0])
assert all(epsilon_greedy(q_ties, 0.0, seed=i) == 0 for i in range(10)), \
    "all Q-values tied: argmax always breaks ties by returning the first index"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def epsilon_greedy(q_values, epsilon, seed=None):
    rng = np.random.RandomState(seed)
    if rng.rand() < epsilon:
        return rng.randint(len(q_values))
    return int(np.argmax(q_values))
```

</details>

### Exercise 3 — Extra practice: full Q-learning training loop (DML #133)

[DML #133 — Implement Q-Learning Algorithm for MDPs](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/133_implement-q-learning-algorithm-for-mdps) asks for the *whole* training loop (not just one TD update) against a general MDP given as explicit transition/reward tensors, matching this exact signature:

```python
q_learning(num_states, num_actions, P, R, terminal_states,
           alpha, gamma, epsilon, num_episodes)
```

`P` has shape `(num_states, num_actions, num_states)` (transition probabilities), `R` has shape `(num_states, num_actions)`. Each episode starts from a random **non-terminal** state and runs until a terminal state is reached, using epsilon-greedy behavior and the standard off-policy TD target.

In [ ]:
def q_learning(num_states, num_actions, P, R, terminal_states,
               alpha, gamma, epsilon, num_episodes):
    """Tabular Q-learning against an explicit MDP (transition tensor P, reward matrix R).
    P: shape (num_states, num_actions, num_states) transition probabilities.
    R: shape (num_states, num_actions) rewards.
    terminal_states: list of terminal state indices.
    Returns: np.ndarray of shape (num_states, num_actions), the learned Q-table.
    """
    # TODO(you): initialize Q to zeros; for num_episodes episodes, start from a
    # random non-terminal state and loop epsilon-greedy action selection ->
    # sample next_state ~ P[state, action] -> TD target (bootstrap only if
    # next_state isn't terminal) -> update Q[state, action] -> until terminal.
    ...

In [ ]:
np.random.seed(42)
P1 = np.array([[[0, 1], [1, 0]], [[1, 0], [1, 0]]])
R1 = np.array([[1, 0], [0, 0]])
Q1_learned = q_learning(2, 2, P1, R1, [1], 0.1, 0.9, 0.1, 10)
assert np.allclose(Q1_learned, [[0.65132156, 0.052902], [0., 0.]], atol=1e-4), \
    "matches the DML #133 fixture exactly (seed=42)"

np.random.seed(42)
P2 = np.array([[[0.5, 0.5], [0, 1]], [[0, 1], [1, 0]]])
R2 = np.array([[0.5, 1], [0, 0]])
Q2_learned = q_learning(2, 2, P2, R2, [1], 0.5, 0.8, 0.2, 5)
assert np.allclose(Q2_learned, [[0.91785477, 0.5], [0., 0.]], atol=1e-4), \
    "matches a second DML #133 fixture with stochastic transitions (seed=42)"

np.random.seed(42)
P3 = np.array([[[1, 0], [0, 1]], [[0.5, 0.5], [0.5, 0.5]]])
R3 = np.array([[2, 1], [0, 0]])
Q3_learned = q_learning(2, 2, P3, R3, [0], 0.3, 0.7, 0.05, 20)
assert np.allclose(Q3_learned, [[0., 0.], [0., 0.]], atol=1e-4), \
    "state 0 is terminal here, so its row of Q never gets an update -- all zeros"

# Edge case: zero reward everywhere -- Q must stay exactly at its zero init,
# no matter how many episodes or how stochastic the transitions are.
Pz = np.array([[[0.5, 0.5], [0.5, 0.5]], [[0.5, 0.5], [0.5, 0.5]]])
Rz = np.zeros((2, 2))
Qz_learned = q_learning(2, 2, Pz, Rz, [1], 0.5, 0.9, 0.3, 25)
assert np.allclose(Qz_learned, 0.0), \
    "zero reward everywhere: every TD target is 0, so Q never leaves its zero initialization"
print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def q_learning(num_states, num_actions, P, R, terminal_states,
               alpha, gamma, epsilon, num_episodes):
    Q = np.zeros((num_states, num_actions))
    non_terminal = [s for s in range(num_states) if s not in set(terminal_states)]
    for _ in range(num_episodes):
        state = np.random.choice(non_terminal)
        while state not in terminal_states:
            if np.random.rand() < epsilon:
                action = np.random.randint(num_actions)
            else:
                action = np.argmax(Q[state])
            next_state = np.random.choice(num_states, p=P[state, action])
            reward = R[state, action]
            if next_state in terminal_states:
                target = reward
            else:
                target = reward + gamma * np.max(Q[next_state])
            Q[state, action] += alpha * (target - Q[state, action])
            state = next_state
    return Q
```

</details>

### Exercise 4 — Extra practice: bandit methods bank (DML #158, #158, #161)

The multi-armed bandit is the simplest RL setting: one state, $k$ actions, no transitions. Three small, classic bandit-method building blocks from DML, grouped into one bank:

- [DML #158 — Epsilon-Greedy Action Selection for n-Armed Bandit](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/158_epsilon-greedy-action-selection-for-n-armed-bandit) — same idea as Exercise 2 above, but with the bandit's own signature `epsilon_greedy(Q, epsilon=0.1)` (no explicit `seed` argument — it draws from the global NumPy RNG).
- [DML #158 — Incremental Mean for Online Reward Estimation](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/158_incremental-mean-for-online-reward-estimation) — update a running mean in **constant memory**: $Q \leftarrow Q + \frac{1}{k}(R - Q)$, no reward history stored.
- [DML #161 — Exponential Weighted Average of Rewards](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/161_exponential-weighted-average-of-rewards) — the same recency-weighting idea, but computed **directly from the closed form** $ (1-\alpha)^k Q_1 + \sum_i \alpha(1-\alpha)^{k-i}R_i $ rather than incrementally, so that more recent rewards count for more.

In [ ]:
def bandit_epsilon_greedy(Q, epsilon=0.1):
    """Epsilon-greedy action selection for an n-armed bandit (DML #158a).
    Q: np.ndarray of estimated action values, shape (n,).
    epsilon: exploration probability.
    Returns: int, the selected action index."""
    # TODO(you): with prob epsilon return a uniformly random action index,
    # otherwise return int(np.argmax(Q))
    ...

def incremental_mean(Q_prev, k, R):
    """Constant-memory running mean update (DML #158b).
    Q_prev: previous mean estimate. k: number of times this action has been
    selected (including the current pull). R: newly observed reward.
    Returns: updated mean estimate, using only Q_prev, k, and R."""
    # TODO(you): Q_prev + (1/k) * (R - Q_prev)
    ...

def exp_weighted_average(Q1, rewards, alpha):
    """Exponential recency-weighted average, computed directly from the closed
    form (DML #161) -- NOT via the incremental update above.
    Q1: initial estimate. rewards: list R_1..R_k. alpha: step size in (0, 1].
    Returns: (1-alpha)^k * Q1 + sum_i alpha*(1-alpha)^(k-i) * R_i."""
    # TODO(you): implement the closed-form weighted sum (a loop or a vectorized
    # sum both work -- k=0 rewards should just return Q1)
    ...

In [ ]:
# --- epsilon-greedy (DML #158a) ---
np.random.seed(0)
assert [bandit_epsilon_greedy(np.array([1, 2, 3]), epsilon=0.0) for _ in range(5)] == [2] * 5, \
    "epsilon=0: always greedy (matches the DML #158 fixture exactly)"

# epsilon=1 always explores uniformly at random. We check the *property* here
# rather than an exact draw sequence: DML's own published fixture for this case
# doesn't reproduce bit-for-bit even from their reference solution, so a brittle
# exact-sequence assert would be testing NumPy RNG internals, not the algorithm.
np.random.seed(1)
draws = [bandit_epsilon_greedy(np.array([5, 2, 1]), epsilon=1.0) for _ in range(300)]
assert set(draws) == {0, 1, 2}, \
    "epsilon=1: every arm should get explored eventually, including the worst one"

# Intermediate epsilon: the greedy arm (index 1) should dominate
np.random.seed(42)
draws2 = [bandit_epsilon_greedy(np.array([1.5, 2.5, 0.5]), epsilon=0.3) for _ in range(300)]
greedy_frac = sum(a == 1 for a in draws2) / len(draws2)
assert greedy_frac > 0.6, \
    "epsilon=0.3: the greedy arm should be picked most of the time"

# Zero-variance Q: all arms tied -- greedy just returns the first argmax
assert bandit_epsilon_greedy(np.array([3.0, 3.0, 3.0]), epsilon=0.0) == 0, \
    "all arms tied: np.argmax breaks ties by returning the first index"

# --- incremental mean (DML #158b) ---
assert abs(incremental_mean(0.0, 1, 5.0) - 5.0) < 1e-9, \
    "first pull (k=1): the mean becomes exactly the first reward"
assert abs(incremental_mean(5.0, 2, 7.0) - 6.0) < 1e-9, \
    "second pull: mean moves halfway toward the new reward"
assert abs(incremental_mean(6.0, 3, 4.0) - 5.3333) < 1e-3, \
    "third pull: mean = 6 + (1/3)*(4-6) = 5.3333"
assert abs(incremental_mean(2.0, 2, 6.0) - 4.0) < 1e-9, \
    "matches the DML #158b worked example: 2.0 + 0.5*(6.0-2.0) = 4.0"

# --- exponential weighted average (DML #161) ---
assert abs(exp_weighted_average(10.0, [4.0, 7.0, 13.0], 0.5) - 10.0) < 1e-6, \
    "matches the DML #161 fixture exactly"
assert abs(exp_weighted_average(0.0, [1.0, 1.0, 1.0, 1.0], 0.1) - 0.3439) < 1e-3, \
    "matches the DML #161 fixture with alpha=0.1"
# Edge case: no rewards observed yet (k=0) -- reduces to the initial estimate
assert abs(exp_weighted_average(3.0, [], 0.5) - 3.0) < 1e-9, \
    "zero observed rewards: the weighted average reduces to the initial estimate Q1"
# alpha=1: only the most recent reward matters, all history is forgotten
assert abs(exp_weighted_average(100.0, [1.0, 2.0, 3.0], 1.0) - 3.0) < 1e-9, \
    "alpha=1: fully recency-weighted, only the last reward survives"
print("✅ Exercise 4 passed")

<details>
<summary>💡 Show solution</summary>

```python
def bandit_epsilon_greedy(Q, epsilon=0.1):
    if np.random.rand() < epsilon:
        return np.random.randint(len(Q))
    return int(np.argmax(Q))

def incremental_mean(Q_prev, k, R):
    return Q_prev + (1.0 / k) * (R - Q_prev)

def exp_weighted_average(Q1, rewards, alpha):
    k = len(rewards)
    total = (1 - alpha) ** k * Q1
    for i, R in enumerate(rewards, start=1):
        total += alpha * (1 - alpha) ** (k - i) * R
    return total
```

</details>